# Pinatubo 1991 — RSAM amplitudes, RSAM trigger counts, and recovered waveforms

Read two legacy binary collections using **FloVoPy** into one multi-channel `rsam` object and one multi-channel `events` object. Compare 10-minute RSAM amplitude, single-channel RSAM STA/LTA trigger counts, and the number of recovered SEISAN waveform files. Generate overview figures for **the complete non-null legacy interval** and **June 1–16, 1991 (inclusive)**.

**Interpretation:** RSAM events are channel-level STA/LTA detections; summing channels counts *triggers*, **not distinct earthquakes**. A waveform file is a recovered recording, not necessarily one earthquake. Waveform inventory is currently limited to **June 1991**: outside June, waveform counts are **unknown (NaN), not zero**. All times UTC; 10-minute bins are left-closed.

**Prerequisites:** FloVoPy's corrected `RSAM.readRSAMbinary()` and `SAM.trim_to_data()`, `get_starttime(non_null=True)`, `get_endtime(non_null=True)`. The notebook runs against your local filesystem, not in this environment.

Generated products use the shared `project_config.json` results root. Source/input files remain separate.


In [ ]:
from pathlib import Path
import sys
_candidates = [Path.cwd(), *Path.cwd().parents]
PROJECT_ROOT = next((p for p in _candidates if (p / 'project_config.json').is_file()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Start in the Pinatubo repository; project_config.json must be available.')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from pinatubo_project import input_path, output_path, runtime_environment
runtime_environment()

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from IPython.display import display
from obspy import UTCDateTime, Trace
from flovopy.core.vdap import fix_legacy_vdap_id
from flovopy.processing.sam import RSAM

LEGACY_DIR = Path('/Users/glennthompson/Library/CloudStorage/Box-Box/thompsong/DATA/pinatubo1991/LEGACY')
RSAM_DIR = LEGACY_DIR / 'BOB' / 'RSAM' / 'RSAM'
RSAMEVENTS_DIR = LEGACY_DIR / 'BOB' / 'RSAM' / 'EVNT' 
FAIR_DIR = Path('/Users/glennthompson/Library/CloudStorage/Box-Box/thompsong/DATA/pinatubo1991/FAIR')
START = UTCDateTime('1991-01-01T00:00:00')
END = UTCDateTime('1991-12-31T23:59:59')  # Reader endpoints are inclusive
SAMPLING_INTERVAL = 600.0  # Legacy archive contains 10-minute values
NETWORK = 'XB'
# Preserve the installed reader's existing default for both collections.
# Verify the historical first-record convention before changing these.
RSAM_SKIP_FIRST_DAY = True
EVENT_SKIP_FIRST_DAY = True
RSAM_CSV_DIR = output_path('20_analysis','legacy_rsam','amplitudes')
EVENT_CSV_DIR = output_path('20_analysis','legacy_rsam','trigger_counts') 
ERUPTION = pd.Timestamp('1991-06-15T00:00:00Z')
JUNE_START = pd.Timestamp('1991-06-01T00:00:00Z')
JULY_START = pd.Timestamp('1991-07-01T00:00:00Z')
JUNE16_END = pd.Timestamp('1991-06-17T00:00:00Z')  # exclusive: includes June 16
SEISAN_TOP = FAIR_DIR / 'SEISAN'
SEISAN_DB = 'PNTBO'
WAV_JUNE_DIR = SEISAN_TOP / 'WAV' / SEISAN_DB / '1991' / '06'
OUTPUT_DIR = output_path('20_analysis','legacy_rsam','figures',create=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
if not RSAM_DIR.is_dir():
    raise FileNotFoundError(f'RSAM directory not found: {RSAM_DIR}')
print('Reading legacy RSAM from:', RSAM_DIR)
print('Saving figures to:', OUTPUT_DIR)

## Binary inventory and ingestion

In [ ]:
def legacy_inventory(folder):
    if not folder.is_dir():
        raise FileNotFoundError(folder)
    paths = sorted(p for p in folder.glob('*91.DAT') if p.is_file())
    return paths, pd.DataFrame([{
        'file': p.name, 'bytes': p.stat().st_size,
        'float32_records': p.stat().st_size // 4,
        'equivalent_days_at_600s': p.stat().st_size / 4 * SAMPLING_INTERVAL / 86400,
    } for p in paths])

rsam_files, rsam_inventory = legacy_inventory(RSAM_DIR)
event_files, event_inventory = legacy_inventory(RSAMEVENTS_DIR)
print(f'RSAM files: {len(rsam_files)}; RSAM EVNT files: {len(event_files)}')
display(rsam_inventory)
display(event_inventory)
if not rsam_files or not event_files:
    raise RuntimeError('One or both legacy binary collections are empty.')

In [ ]:
def load_legacy_collection(paths, metric='mean', **reader_options):
    """Read each legacy binary file via FloVoPy; return one multi-channel RSAM."""
    frames, failures = {}, {}
    for path in paths:
        try:
            part = RSAM.readRSAMbinary(
                filepath=str(path), stime=START, etime=END,
                sampling_interval=SAMPLING_INTERVAL,
                fill_missing_with=None, verbose=False, **reader_options,
            )
            if not part.dataframes:
                raise ValueError('No dataframes returned')
            # Decode the filename station directly: the installed reader's
            # conversion option currently passes unsupported add_T=False.
            if len(part.dataframes) != 1:
                raise ValueError('Expected one channel per legacy binary file')
            if not path.stem.upper().endswith('91'):
                raise ValueError(f'Expected a 1991 filename: {path.name}')
            legacy_station = path.stem[:-2].upper()
            tr = Trace()
            tr.stats.station = legacy_station
            fix_legacy_vdap_id(tr, network=NETWORK)
            trace_id = tr.id
            for df in part.dataframes.values():
                if trace_id in frames:
                    raise ValueError(f'Duplicate trace ID: {trace_id}')
                if not {'time', 'mean'} <= set(df.columns):
                    raise ValueError(f'{trace_id}: expected time and mean')
                df = df.copy(deep=True)
                if metric != 'mean':
                    df = df.rename(columns={'mean': metric})
                frames[trace_id] = df
        except Exception as exc:
            failures[path.name] = f'{type(exc).__name__}: {exc}'
    if failures:
        display(pd.DataFrame([{'file': k, 'error': v} for k, v in failures.items()]))
        raise RuntimeError(f'{len(failures)} legacy files could not be read; resolve before analysis')
    if not frames:
        raise RuntimeError('No legacy data loaded')
    return RSAM(dataframes=frames, sampling_interval=SAMPLING_INTERVAL)

# Set first-record handling explicitly for both binary collections.
rsam = load_legacy_collection(rsam_files, skip_first_day=RSAM_SKIP_FIRST_DAY)
events = load_legacy_collection(
    event_files, metric='event_count', skip_first_day=EVENT_SKIP_FIRST_DAY,
)

for label, obj, metric in [('RSAM', rsam, 'mean'), ('EVNT', events, 'event_count')]:
    print(f'{label}: {len(obj.dataframes)} channels; '
          f'full {obj.starttime} – {obj.endtime}; '
          f'finite {obj.get_starttime(non_null=True, metric=metric)} – '
          f'{obj.get_endtime(non_null=True, metric=metric)}')
    print('  finite samples:', sum(np.isfinite(pd.to_numeric(df[metric], errors='coerce')).sum()
                                   for df in obj.dataframes.values()))

## Validation: event counts and time alignment

Zeros are valid no-trigger intervals. Check that the EVNT measurements are integer-valued, nonnegative, and on the expected 600-second grid. Both collections explicitly use skip_first_day=True, preserving the installed reader's previous effective behavior. Compare time axes and verify the historical first-record convention before interpreting cross-series timing.

In [ ]:
def summarize_channels(obj, metric):
    rows = []
    for tid, df in sorted(obj.dataframes.items()):
        t = pd.to_numeric(df['time'], errors='coerce').to_numpy(dtype=float)
        x = pd.to_numeric(df[metric], errors='coerce').to_numpy(dtype=float)
        ok = np.isfinite(t) & np.isfinite(x)
        rows.append({'trace_id': tid, 'rows': len(df), 'finite': int(ok.sum()),
                     'zeros': int((ok & (x == 0)).sum()),
                     'first_finite': pd.to_datetime(t[ok].min(), unit='s', utc=True) if ok.any() else pd.NaT,
                     'last_finite': pd.to_datetime(t[ok].max(), unit='s', utc=True) if ok.any() else pd.NaT,
                     'max': float(x[ok].max()) if ok.any() else np.nan})
    return pd.DataFrame(rows)

display(summarize_channels(rsam, 'mean'))
display(summarize_channels(events, 'event_count'))
all_counts = np.concatenate([pd.to_numeric(df['event_count'], errors='coerce').to_numpy(dtype=float)
                             for df in events.dataframes.values()])
finite_counts = all_counts[np.isfinite(all_counts)]
assert finite_counts.size, 'No finite event counts'
print('EVNT integer-valued:', bool(np.all(np.isclose(finite_counts, np.rint(finite_counts)))))
print('EVNT nonnegative:', bool(np.all(finite_counts >= 0)))
assert np.all(finite_counts >= 0) and np.all(np.isclose(finite_counts, np.rint(finite_counts))),     'Unexpected EVNT values: inspect file format before treating as counts'
common_ids = sorted(set(rsam.dataframes) & set(events.dataframes))
print('Matching trace IDs:', len(common_ids), 'of', len(rsam.dataframes), 'RSAM and', len(events.dataframes), 'EVNT')
for tid in common_ids[:5]:
    a = pd.to_numeric(rsam.dataframes[tid]['time'], errors='coerce').dropna().to_numpy()
    b = pd.to_numeric(events.dataframes[tid]['time'], errors='coerce').dropna().to_numpy()
    print(tid, 'first timestamp difference (EVNT - RSAM, seconds):', b[0]-a[0])

## Export ingested measurements

Save yearly channel CSVs under separate FAIR folders for amplitudes and trigger counts. These use XB IDs and preserve low gain (PI2L → XB.PI2..ELZ). Source binary files are unchanged. Existing yearly CSVs are merged by timestamp using RSAM.write() defaults.


In [ ]:
for obj, folder in [(rsam, RSAM_CSV_DIR), (events, EVENT_CSV_DIR)]:
    folder.mkdir(parents=True, exist_ok=True)
    obj.write(str(folder), ext='csv', overwrite=False)
    print('Saved channel CSVs to:', folder)


## Network-wide trimming (without changing the source objects)

In [ ]:
rsam_trimmed = rsam.trim_to_data(metric='mean', per_station=False, inplace=False)
events_trimmed = events.trim_to_data(metric='event_count', per_station=False, inplace=False)

starts = [rsam.get_starttime(non_null=True, metric='mean'),
          events.get_starttime(non_null=True, metric='event_count')]
ends = [rsam.get_endtime(non_null=True, metric='mean'),
        events.get_endtime(non_null=True, metric='event_count')]
full_start = pd.Timestamp(min(starts).datetime, tz='UTC').floor('D')
full_end = pd.Timestamp(max(ends).datetime, tz='UTC').ceil('D')
print('Combined finite-data plot interval:', full_start, 'to', full_end, '(exclusive)')
print('Original RSAM extent unchanged:', rsam.starttime, rsam.endtime)

## Ten-minute network time series

**RSAM network median** is the median amplitude among channels reporting finite measurements in a bin (a robust network summary, not a physically additive amplitude). **Network trigger total** sums all finite channel trigger counts; the channel count is plotted alongside it so changes in network availability are visible. Bins with no reporting channels remain NaN, not zero.

In [ ]:
def channel_matrix(obj, metric, grid):
    columns = {}
    for tid, df in sorted(obj.dataframes.items()):
        t = pd.to_datetime(pd.to_numeric(df['time'], errors='coerce'), unit='s', utc=True)
        x = pd.to_numeric(df[metric], errors='coerce')
        good = t.notna() & np.isfinite(x)
        if not good.any():
            continue
        s = pd.Series(x.loc[good].to_numpy(dtype=float), index=pd.DatetimeIndex(t.loc[good]))
        # Group only if repeated timestamps exist; never sum duplicate amplitudes.
        s = s.groupby(level=0).mean()
        columns[tid] = s.reindex(grid)
    return pd.DataFrame(columns, index=grid)

grid = pd.date_range(full_start, full_end, freq='10min', inclusive='left', tz='UTC')
rsam_matrix = channel_matrix(rsam_trimmed, 'mean', grid)
event_matrix = channel_matrix(events_trimmed, 'event_count', grid)
network = pd.DataFrame(index=grid)
network.index.name = 'time_utc'
network['rsam_median'] = rsam_matrix.median(axis=1, skipna=True)
network['rsam_p90'] = rsam_matrix.quantile(0.90, axis=1)
network['rsam_channels'] = rsam_matrix.notna().sum(axis=1)
network['trigger_total'] = event_matrix.sum(axis=1, min_count=1)
network['trigger_channels'] = event_matrix.notna().sum(axis=1)
network['trigger_channels_positive'] = (event_matrix > 0).sum(axis=1)
print('10-minute bins:', len(network))
display(network.loc[JUNE_START:JUNE_START + pd.Timedelta(minutes=40)])

## SEISAN waveform inventory — June 1991

The files use the **`M.`** SEISAN naming convention (e.g. `1991-06-01-0017-11M.PNTBO_008`), rather than the `S.` marker assumed by the current `SeisanArchive._wavpath2datetime`. The companion patched `seisanarchive.py` accepts both markers. The WAV database is at `SEISAN_TOP / 'WAV' / 'PNTBO' / YYYY / MM`.

Count waveform **files by filename start time** in ten-minute UTC bins; these are not unique earthquake counts. Only June has been inventoried, so other months remain NaN in the network comparison.

In [ ]:
from flovopy.seisanio.core.seisanarchive import SeisanArchive

archive = SeisanArchive(root=SEISAN_TOP, db_cont=SEISAN_DB)

# The installed SeisanArchive must support both M. and S. filename markers.
# Apply the companion seisanarchive.py patch if the following test fails.
example_name = '1991-06-01-0017-11M.PNTBO_008'
try:
    example_time = archive._wavpath2datetime(example_name)
except (ValueError, IOError) as exc:
    raise RuntimeError(
        'SeisanArchive._wavpath2datetime must accept M. files. '
        'Install the companion patched seisanarchive.py first.'
    ) from exc
assert example_time == UTCDateTime('1991-06-01T00:17:11Z')

if not WAV_JUNE_DIR.is_dir():
    raise FileNotFoundError(f'June waveform directory not found: {WAV_JUNE_DIR}')

# Inventory all actual files (including malformed names, for audit).
waveform_paths = sorted(
    p for p in WAV_JUNE_DIR.iterdir()
    if p.is_file() and not p.name.startswith('.')
)

def parse_seisan_start(path):
    try:
        return pd.Timestamp(archive._wavpath2datetime(path).datetime, tz='UTC')
    except (ValueError, IOError, OverflowError):
        return pd.NaT

waveform_inventory = pd.DataFrame(
    [{'filename': p.name, 'path': str(p), 'start_utc': parse_seisan_start(p)}
     for p in waveform_paths],
    columns=['filename', 'path', 'start_utc'],
)
unparsed = waveform_inventory[waveform_inventory['start_utc'].isna()]
print(f'June files found: {len(waveform_inventory)}; '
      f'parseable: {len(waveform_inventory) - len(unparsed)}; '
      f'unparsed: {len(unparsed)}')
if not unparsed.empty:
    display(unparsed.head(30))
    print('WARNING: Unparsed filenames excluded from waveform counts.')

parsed = waveform_inventory.dropna(subset=['start_utc']).copy()
parsed = parsed.loc[
    (parsed['start_utc'] >= JUNE_START) & (parsed['start_utc'] < JULY_START)
].copy()
parsed['bin_10min'] = parsed['start_utc'].dt.floor('10min')
waveform_counts = parsed.groupby('bin_10min').size()

# Zero-fill only the inventoried month; do not equate uninventoried months with zero.
june_grid = grid[(grid >= JUNE_START) & (grid < JULY_START)]
network['waveform_files'] = np.nan
network.loc[june_grid, 'waveform_files'] = (
    waveform_counts.reindex(june_grid, fill_value=0).to_numpy()
)
print('Waveform file starts in June grid:',
      int(network.loc[june_grid, 'waveform_files'].sum()))
assert len(parsed) == int(network.loc[june_grid, 'waveform_files'].sum()), (
    'Parsed June waveform count does not match 10-minute bins; check grid bounds.'
)
display(parsed.head())

## Overview comparison — entire non-null interval and June 1–16

In [ ]:
def comparison_plot(start, end, label, filename):
    view = network.loc[(network.index >= start) & (network.index < end)]
    fig, axes = plt.subplots(4, 1, figsize=(17, 11), sharex=True, constrained_layout=True)
    axes[0].plot(view.index, view['rsam_median'], lw=0.7, label='Median RSAM across reporting channels')
    axes[0].plot(view.index, view['rsam_p90'], lw=0.6, alpha=0.65, label='90th percentile RSAM')
    axes[0].set_ylabel('RSAM (legacy units)'); axes[0].legend(loc='upper left')
    axes[1].plot(view.index, view['trigger_total'], lw=0.7)
    axes[1].set_ylabel('STA/LTA triggers / 10 min (sum)')
    axes[2].step(view.index, view['waveform_files'], where='post', lw=0.8)
    axes[2].set_ylabel('Waveform files started / 10 min')
    axes[2].text(0.01, 0.92, 'Outside June: not inventoried (blank ≠ zero)',
                 transform=axes[2].transAxes, fontsize=9, va='top')
    axes[3].plot(view.index, view['rsam_channels'], lw=0.8, label='RSAM reporting channels')
    axes[3].plot(view.index, view['trigger_channels'], lw=0.8, label='EVNT reporting channels')
    axes[3].set_ylabel('Reporting channels'); axes[3].legend(loc='upper left')
    for ax in axes:
        ax.axvline(ERUPTION, color='tab:red', ls='--', lw=1.1)
        ax.grid(alpha=0.2)
        ax.set_xlim(start, end)
    axes[-1].xaxis.set_major_formatter(mdates.DateFormatter('%b %d', tz=mdates.UTC))
    axes[-1].xaxis.set_major_locator(mdates.DayLocator(interval=2) if end-start <= pd.Timedelta(days=20)
                                         else mdates.MonthLocator())
    fig.suptitle(f'Pinatubo 1991 — RSAM, channel triggers, and SEISAN WAV files: {label}')
    fig.savefig(OUTPUT_DIR / filename, dpi=170)
    plt.show()

comparison_plot(full_start, full_end, 'complete finite-data interval', 'rsam_events_waveforms_full.png')
comparison_plot(JUNE_START, JUNE16_END, 'June 1–16 inclusive', 'rsam_events_waveforms_june01_16.png')

## Channel-by-channel RSAM and trigger plots (both intervals)

In [ ]:
def station_stack(matrix, start, end, title, ylabel, filename):
    view = matrix.loc[(matrix.index >= start) & (matrix.index < end)]
    ids = [c for c in view if view[c].notna().any()]
    if not ids:
        print('No data for', title); return
    fig, axes = plt.subplots(len(ids), 1, figsize=(17, max(6, 1.4 * len(ids))),
                             sharex=True, constrained_layout=True)
    axes = np.atleast_1d(axes)
    for ax, tid in zip(axes, ids):
        ax.plot(view.index, view[tid], lw=0.65)
        ax.set_ylabel(tid, rotation=0, ha='right', va='center')
        ax.axvline(ERUPTION, color='tab:red', ls='--', lw=0.9)
        ax.grid(alpha=0.15)
        ax.set_xlim(start, end)
    axes[0].set_title(f'{title} — {ylabel}')
    axes[-1].xaxis.set_major_formatter(mdates.DateFormatter('%b %d', tz=mdates.UTC))
    fig.savefig(OUTPUT_DIR / filename, dpi=150)
    plt.show()

for begin, finish, tag in [(full_start, full_end, 'full'),
                           (JUNE_START, JUNE16_END, 'june01_16')]:
    station_stack(rsam_matrix, begin, finish, 'RSAM by channel', 'legacy amplitude', f'rsam_channels_{tag}.png')
    station_stack(event_matrix, begin, finish, 'RSAM STA/LTA triggers by channel', 'counts / 10 min',
                  f'events_channels_{tag}.png')

## Daily completeness and daily totals

In [ ]:
def availability_matrix(matrix):
    # 144 expected 10-minute bins per UTC day, including zero-valued observations.
    return matrix.notna().resample('D').sum() / (86400 / SAMPLING_INTERVAL)

for label, matrix in [('RSAM', rsam_matrix), ('EVNT', event_matrix)]:
    avail = availability_matrix(matrix)
    fig, ax = plt.subplots(figsize=(16, max(4, 0.45 * len(avail.columns) + 1)), constrained_layout=True)
    im = ax.imshow(avail.T.to_numpy(), aspect='auto', interpolation='nearest', vmin=0, vmax=1,
                   extent=[mdates.date2num(avail.index[0]),
                           mdates.date2num(avail.index[-1] + pd.Timedelta(days=1)),
                           len(avail.columns)-0.5, -0.5])
    ax.set_yticks(np.arange(len(avail.columns)), avail.columns)
    ax.xaxis_date(); ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %d', tz=mdates.UTC))
    ax.xaxis.set_major_locator(mdates.MonthLocator())
    ax.axvline(ERUPTION, color='white', ls='--')
    ax.set_title(f'{label} — fraction of expected 10-minute records with finite measurements')
    fig.colorbar(im, ax=ax, label='Daily finite fraction')
    fig.savefig(OUTPUT_DIR / f'{label.lower()}_availability_heatmap.png', dpi=170)
    plt.show()

daily = pd.DataFrame(index=network.resample('D').size().index)
daily['trigger_sum'] = network['trigger_total'].resample('D').sum(min_count=1)
daily['waveform_file_count'] = network['waveform_files'].resample('D').sum(min_count=1)
daily['rsam_median'] = network['rsam_median'].resample('D').median()
daily['rsam_channels_max'] = network['rsam_channels'].resample('D').max()
daily['event_channels_max'] = network['trigger_channels'].resample('D').max()
display(daily.loc['1991-06-01':'1991-06-16'])

## June 1–16: daily triggers versus waveform-file starts

In [ ]:
view = daily.loc['1991-06-01':'1991-06-16']
fig, axes = plt.subplots(2, 1, figsize=(15, 7), sharex=True, constrained_layout=True)
axes[0].bar(view.index, view['trigger_sum'], width=0.8)
axes[0].set_ylabel('Channel STA/LTA triggers / day')
axes[1].bar(view.index, view['waveform_file_count'], width=0.8)
axes[1].set_ylabel('SEISAN WAV files started / day')
for ax in axes:
    ax.axvline(ERUPTION, color='tab:red', ls='--')
    ax.grid(alpha=0.2, axis='y')
axes[1].xaxis.set_major_formatter(mdates.DateFormatter('%b %d', tz=mdates.UTC))
fig.suptitle('June 1–16, 1991 — network trigger counts vs recovered waveform files')
fig.savefig(OUTPUT_DIR / 'daily_triggers_vs_waveform_files_june01_16.png', dpi=170)
plt.show()

## Interpretation and next checks

- **Network trigger sum is not an earthquake catalog.** One physical event can trigger multiple channels, and a single channel can retrigger. No coincidence association or deduplication is applied.
- **Waveform files** are counted by SEISAN filename start time. One file can hold multiple traces; one event may generate multiple files, and not every trigger results in an archived waveform.
- **Temporal alignment:** verify whether EVNT files and amplitude RSAM use identical first-record offsets and whether time tags denote interval start or end. The notebook reports first-timestamp offsets but cannot independently establish the original system's timestamp convention.
- **Coverage:** NaN indicates no recovered measurement; zero event count is a valid observation. A channel's event total is only comparable across periods with similar channel availability. The bottom overview panel shows reporting channel counts.
- **Waveform inventory:** only `FAIR_DIR / 'SEISAN' / 'WAV' / '1991' / '06'` is scanned. Blank waveform curves outside June indicate **not inventoried**, not no files. Review any unparsed filename list.
- **RSAM amplitudes:** network median/p90 are descriptive channel summaries; heterogeneous instrument gains or response calibrations may make cross-channel amplitude comparisons imperfect.
- **No binary files are altered.** All figures are saved to `OUTPUT_DIR`; source `rsam` and `events` objects remain available for further exploration.